<a href="https://colab.research.google.com/github/MayerT1/LiDAR_Dev/blob/main/Multi_branch_DL_approach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rasterio geopandas pandas torch torchvision scikit-learn einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [2]:

import os
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
import pandas as pd
import rasterio
from rasterio.windows import Window
from torchvision import transforms
import numpy as np
from torch.utils.data import Dataset, DataLoader
from einops import rearrange


In [3]:
class MultiSensorGEDIDataset(Dataset):
    def __init__(self, csv_path, tif_dir, sensors, target_col='rh100', patch_size_m=64):
        self.df = pd.read_csv(csv_path)
        self.tif_dir = tif_dir
        self.sensors = sensors  # e.g., {'s1': 10, 's2': 20, 'landsat': 30}
        self.target_col = target_col
        self.patch_size_m = patch_size_m

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        lat, lon, tile_id = row['lat'], row['lon'], row['block_id']
        label = row[self.target_col]

        sensor_patches = []
        for sensor, res in self.sensors.items():
            tif_path = os.path.join(self.tif_dir, f"{sensor}_{row['split']}_{tile_id}.tif")
            with rasterio.open(tif_path) as src:
                # Convert lat/lon to pixel
                x, y = src.index(lon, lat)
                rad = int(self.patch_size_m / res // 2)
                window = Window(x - rad, y - rad, rad * 2, rad * 2)

                patch = src.read(window=window)
                patch = patch.astype(np.float32)
                patch = np.nan_to_num(patch)
                sensor_patches.append(torch.tensor(patch))

        return torch.stack(sensor_patches), torch.tensor(label, dtype=torch.float32)


In [4]:
class FusionCNN(nn.Module):
    def __init__(self, sensor_channels, patch_size):
        super().__init__()
        self.branches = nn.ModuleDict()
        for name, in_ch in sensor_channels.items():
            self.branches[name] = nn.Sequential(
                nn.Conv2d(in_ch, 16, 3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, 3, padding=1),
                nn.ReLU()
            )

        self.upsample_to = patch_size // 2  # assuming 2x downsample above

        self.head = nn.Sequential(
            nn.Conv2d(32 * len(sensor_channels), 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 1)
        )

    def forward(self, x):  # x = (N, S, C, H, W)
        features = []
        for i, (name, branch) in enumerate(self.branches.items()):
            xi = x[:, i]
            fi = branch(xi)
            fi = nn.functional.interpolate(fi, size=(self.upsample_to, self.upsample_to), mode='bilinear')
            features.append(fi)
        x = torch.cat(features, dim=1)
        return self.head(x).squeeze()


In [6]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [8]:
from torch.utils.data import random_split

# Settings
sensors = {'dem': 1, 's1': 2, 's2': 3, 'landsat': 6}  # adapt based on bands
sensor_res = {'dem': 30, 's1': 10, 's2': 20, 'landsat': 30}

dataset = MultiSensorGEDIDataset(
    csv_path='/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/target_data/filtered_csvs/GEDI_filtered_merged.csv',
    tif_dir='/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data',
    sensors=sensor_res
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)

# Model
model = FusionCNN(sensor_channels=sensors, patch_size=64)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training Loop
for epoch in range(10):
    model.train()
    for x, y in train_loader:
        y_hat = model(x)
        loss = criterion(y_hat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = sum(criterion(model(x), y).item() for x, y in val_loader) / len(val_loader)
    print(f"Epoch {epoch+1}: val_loss = {val_loss:.4f}")


RuntimeError: stack expects each tensor to be equal size, but got [2, 0, 0] at entry 0 and [4, 0, 0] at entry 1